# Cleaning a Census Table

**Series:** Cleaning and Preparing Data in Python (notebook 1 of 2)
**Time:** about 35 minutes

**In this notebook you will:**
- Inspect a raw table before changing it
- Keep identifiers as text so leading zeros survive, and build a GEOID
- Turn the Census Bureau's special codes into missing values
- Judge how reliable each estimate is from its margin of error
- Compute a rate safely and check the result with a join and a map

**You should already know:** pandas basics (Foundations notebook <a href="https://colab.research.google.com/github/SpatialTurn/Foundations/blob/main/notebooks/03a_Working_with_Tables_in_pandas.ipynb">3a</a>) and table joins (notebook <a href="https://colab.research.google.com/github/SpatialTurn/Foundations/blob/main/notebooks/04c_Joining_Tables_and_Layers.ipynb">4c</a>).

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## The raw table

The file `acs_2024_tippecanoe_tracts_raw.csv` is exactly what the Census Bureau's API returned for the 43 census
tracts of Tippecanoe County, Indiana (2020 to 2024 American Community Survey, 5-year estimates). Nothing has been
fixed. It holds four estimates, each with its **margin of error**:

| Column | Meaning |
|---|---|
| `B01003_001E`, `B01003_001M` | Total population: estimate (E) and margin of error (M) |
| `B19013_001E`, `B19013_001M` | Median household income (dollars) |
| `B25064_001E`, `B25064_001M` | Median gross rent (dollars per month) |
| `B08201_001E`, `B08201_001M` | Households |
| `B08201_002E`, `B08201_002M` | Households with no vehicle |
| `state`, `county`, `tract` | The parts of the tract's identifier |

## Step 1: Inspect before you change anything

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

URL = "https://raw.githubusercontent.com/SpatialTurn/DataProcessing/main/data/acs_2024_tippecanoe_tracts_raw.csv"
raw = pd.read_csv(URL)
print(raw.shape)
raw.head()

In [ ]:
raw.dtypes

Two problems are already visible:

- **The tract codes lost their leading zeros.** The API sent `000100`, but pandas saw digits and read a number,
  `100`. The tract identifier is a *code*, not a quantity: it must stay text.
- **Some values are huge negative numbers.** Look at the minimum of each column:

In [ ]:
raw.describe().loc[["min", "max"]].T

No tract has an income of minus 666 million dollars. These are the Census Bureau's **annotation codes**: special
values that stand for "no estimate" in a column that must hold numbers.

| Code | In an estimate (E) column | In a margin of error (M) column |
|---|---|---|
| `-666666666` | Could not be computed (too few sample cases, or no households) | |
| `-999999999` | Cannot be shown (too few sample cases) | Cannot be shown |
| `-888888888` | Not applicable | Not applicable |
| `-222222222` | | Could not be computed (too few sample observations) |
| `-333333333` | | Could not be computed: the median is in an open-ended range (income `250001` means "250,000 or more") |
| `-555555555` | | The estimate is controlled (exact), so the margin is zero |

Averaging these codes as numbers would wreck any statistic. Find the rows that contain them:

In [ ]:
codes = raw.select_dtypes("number").lt(-100_000_000)
print(codes.sum()[codes.sum() > 0])
raw.loc[codes.any(axis=1), ["NAME", "B01003_001E", "B19013_001E", "B25064_001E", "B08201_001E"]]

Three tracts. Census Tract 104 has 8,248 residents but **no households**: it is the Purdue residence halls,
where everyone lives in group quarters, so median household income and median rent do not exist. Tracts 16.01
and 105 have too few renters or households in the sample for one of the medians.

## Step 2: Read the identifiers as text and build a GEOID

Read the file again, telling pandas which columns are text. (Fixing `100` afterwards with `zfill` also works,
but reading correctly is safer: it never loses information in the first place.)

In [ ]:
raw = pd.read_csv(URL, dtype={"state": str, "county": str, "tract": str})
raw["GEOID"] = raw["state"] + raw["county"] + raw["tract"]
raw[["NAME", "state", "county", "tract", "GEOID"]].head(3)

An 11-character GEOID: 2 for the state (`18`, Indiana), 3 for the county (`157`, Tippecanoe) and 6 for the
tract (`000100`, tract 1). This is the key that joins the table to the tract boundaries.

## Step 3: Rename the columns

Variable codes are precise but unreadable. Rename them once, with a dictionary you can reuse.

In [ ]:
NAMES = {
    "B01003_001E": "pop", "B01003_001M": "pop_moe",
    "B19013_001E": "median_income", "B19013_001M": "median_income_moe",
    "B25064_001E": "median_rent", "B25064_001M": "median_rent_moe",
    "B08201_001E": "households", "B08201_001M": "households_moe",
    "B08201_002E": "hh_no_vehicle", "B08201_002M": "hh_no_vehicle_moe",
}
acs = raw.rename(columns=NAMES)
acs["name"] = acs["NAME"].str.split("; ").str[0]          # "Census Tract 1; Tippecanoe County; Indiana"
acs = acs[["GEOID", "name"] + list(NAMES.values())]
acs.head(3)

## Step 4: Turn the codes into missing values

Replace every annotation code with `NaN` (pandas' missing value). Count before and after, so you know exactly
what changed.

In [ ]:
num = acs.select_dtypes("number").columns
print("codes before:", acs[num].lt(-100_000_000).sum().sum())

acs[num] = acs[num].mask(acs[num] < -100_000_000)

print("codes after: ", acs[num].lt(-100_000_000).sum().sum())
print("missing values now:")
print(acs[num].isna().sum()[acs[num].isna().sum() > 0])

Eight values are now missing: median income and its margin in tracts 104 and 105, and median rent and its
margin in tracts 16.01 and 104. Note that pandas changed those four columns from whole numbers to decimals: a
whole-number column cannot hold `NaN`.

## Step 5: How reliable is each estimate?

ACS numbers are **estimates from a sample**, and the margin of error (MOE) says how far off they might be: the
true value lies within estimate plus or minus MOE with 90% confidence. A common way to compare reliability is the
**coefficient of variation (CV)**:

$$CV = \frac{MOE / 1.645}{\text{estimate}} \times 100$$

(1.645 turns a 90% margin into a standard error.) A widely used rule of thumb: CV under 12% is **high**
reliability, 12 to 40% **medium**, over 40% **low**; treat low-reliability estimates with great care.

In [ ]:
def cv(est, moe):
    """Coefficient of variation in percent; NaN where the estimate is missing or zero."""
    return (moe / 1.645) / est.where(est > 0) * 100

def reliability(c):
    return pd.cut(c, [0, 12, 40, np.inf], labels=["high", "medium", "low"], right=False)

acs["income_cv"] = cv(acs["median_income"], acs["median_income_moe"])
acs["no_vehicle_cv"] = cv(acs["hh_no_vehicle"], acs["hh_no_vehicle_moe"])

print("Median income:", reliability(acs["income_cv"]).value_counts().to_dict())
print("Households without a car:", reliability(acs["no_vehicle_cv"]).value_counts().to_dict())

Median income is reliable enough in every tract. The count of **households without a car is not**: in most
tracts it is a small number (under 100 households in 19 tracts), and small counts from a sample have large relative errors. This
does not make the data useless, but it changes how you use it: report margins of error, combine tracts into
larger areas, or describe broad patterns rather than ranking single tracts.

## Step 6: Compute a rate safely

The share of households without a car is `hh_no_vehicle / households`. Tract 104 has **zero** households, and
dividing by zero gives an infinite or undefined value. Make the rate missing there instead.

In [ ]:
acs["pct_no_vehicle"] = (acs["hh_no_vehicle"] / acs["households"].where(acs["households"] > 0) * 100).round(1)
print(acs["pct_no_vehicle"].describe().round(1))
print("Missing:", acs.loc[acs["pct_no_vehicle"].isna(), "name"].tolist())

## Step 7: Check the result

Two checks: does every row join to a tract boundary, and does the table agree with the file the Foundations
module uses (prepared from the same download)? That file keeps the code -666666666 on purpose, for the
Foundations cleaning lesson, so clean its income column first.

In [ ]:
tracts = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_tracts.geojson")
check = tracts.merge(acs, on="GEOID", how="outer", indicator=True)
print(check["_merge"].value_counts())

In [ ]:
clean = pd.read_csv("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_acs_2024.csv", dtype={"GEOID": str})
clean["median_income"] = clean["median_income"].mask(clean["median_income"] < 0)
both = acs.merge(clean, on="GEOID", suffixes=("", "_foundations"))
same = (both["median_income"].fillna(-1) == both["median_income_foundations"].fillna(-1))
print(same.sum(), "of", len(both), "tracts have the same median income")

All 43 tracts match on both checks. A quick map is the last check: are the missing values where you expect them,
and does the pattern make sense?

In [ ]:
gdf = tracts.merge(acs, on="GEOID")
ax = gdf.plot(column="median_income", cmap="Greens", legend=True, edgecolor="white", linewidth=0.4,
              missing_kwds={"color": "lightgray", "hatch": "///", "label": "No estimate"},
              legend_kwds={"label": "Median household income ($)"}, figsize=(7, 7))
ax.set_title("Median household income, 2020 to 2024 (gray: no estimate)")
ax.set_axis_off()

The hatched tracts are 104 (the Purdue residence halls) and 105. Keep the cleaned table for the next
steps of a project; in Colab, save it to the session and download it from the Files panel.

In [ ]:
acs.to_csv("tippecanoe_acs_2024_clean.csv", index=False)

## Try it yourself

Clean the **median rent** column the same way and judge its reliability.

1. How many tracts have no rent estimate?
2. How many rent estimates have high, medium and low reliability?

*Hint:* Step 4 already turned the codes into NaN; reuse `cv()` and `reliability()` from Step 5.

<details>
<summary><b>Show a solution</b></summary>

```python
print(acs["median_rent"].isna().sum(), "tracts without a rent estimate")
acs["rent_cv"] = cv(acs["median_rent"], acs["median_rent_moe"])
print(reliability(acs["rent_cv"]).value_counts().to_dict())
```

</details>

In [ ]:
# Write your answer here

## Recap

- Inspect first: shape, types, minimum and maximum reveal most problems
- Read identifiers as text (`dtype=str`) so leading zeros survive; build GEOID from its parts
- Census annotation codes (such as -666666666) are not numbers: turn them into `NaN` and count what changed
- Margins of error matter: the coefficient of variation shows which estimates are too uncertain to rank
- Guard rates against division by zero, then check with a join (`indicator=True`) and a map

**Next:** [01b Cleaning Points and Polygons](https://colab.research.google.com/github/SpatialTurn/DataProcessing/blob/main/notebooks/01b_Cleaning_Points_and_Polygons.ipynb)